In [2]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.base import BaseEstimator, ClassifierMixin

# Load dataset
df = pd.read_csv('learning_disability_dataset.csv')

# Encode categorical features
df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
df['Sleep Quality'] = df['Sleep Quality'].map({'Poor': 0, 'Average': 1, 'Good': 2})

# Features & Labels
X = df.drop(columns=['Labels'])
y_raw = df['Labels'].apply(
    lambda x: [
        label.strip().capitalize()
        for label in str(x).split(',')
        if label.strip().lower() not in ['nan', '', 'none']
    ]
)

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(y_raw)

# Standardize features for deep learning model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save scaler
os.makedirs('saved_models', exist_ok=True)
joblib.dump(scaler, 'saved_models/scaler.pkl')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Define deep learning model
def create_nn_model():
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(Dropout(0.3))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(y_train.shape[1], activation='sigmoid'))  # sigmoid for multi-label
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Custom wrapper for Keras model
class KerasMultiOutputClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, build_fn, epochs=50, batch_size=16, verbose=0):
        self.build_fn = build_fn
        self.epochs = epochs
        self.batch_size = batch_size
        self.verbose = verbose

    def fit(self, X, y):
        self.model = self.build_fn()
        self.model.fit(
            X, y,
            epochs=self.epochs,
            batch_size=self.batch_size,
            verbose=self.verbose,
            callbacks=[EarlyStopping(patience=5, restore_best_weights=True)]
        )
        return self

    def predict(self, X):
        pred = self.model.predict(X)
        return (pred > 0.5).astype(int)

# Define all models
models = {
    'random_forest': MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42)),
    'svm': MultiOutputClassifier(SVC(probability=True, kernel='linear', random_state=42)),
    'logistic_regression': MultiOutputClassifier(LogisticRegression(max_iter=1000, random_state=42)),
    'knn': MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5)),
    'gradient_boosting': MultiOutputClassifier(GradientBoostingClassifier(n_estimators=100, random_state=42)),
    'deep_learning': KerasMultiOutputClassifier(create_nn_model, epochs=30, batch_size=16, verbose=1)
}

# Train and evaluate
for name, model in models.items():
    print(f"\n🔹 Training {name} model...")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_pred = np.nan_to_num(y_pred, nan=0.0).astype(int)

    print(f"📊 Classification Report for {name}:")
    try:
        print(classification_report(
            y_test,
            y_pred,
            target_names=mlb.classes_,
            labels=range(len(mlb.classes_))
        ))
    except ValueError as e:
        print("⚠️ Report issue:", str(e))

    # Save traditional models (not for deep learning)
    if name != "deep_learning":
        joblib.dump(model, f'saved_models/{name}_model.pkl')
    else:
        model.model.save("saved_models/deep_learning_model.h5")

# Save label binarizer
joblib.dump(mlb, 'saved_models/mlb.pkl')

print("\n✅ All models trained and saved.")



🔹 Training random_forest model...
📊 Classification Report for random_forest:
              precision    recall  f1-score   support

        Adhd       0.89      0.83      0.86        70
    Dyslexia       0.99      0.84      0.91        97

   micro avg       0.95      0.83      0.89       167
   macro avg       0.94      0.83      0.88       167
weighted avg       0.95      0.83      0.89       167
 samples avg       0.57      0.53      0.54       167



c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver


🔹 Training svm model...
📊 Classification Report for svm:
              precision    recall  f1-score   support

        Adhd       0.83      0.83      0.83        70
    Dyslexia       0.90      0.84      0.87        97

   micro avg       0.87      0.83      0.85       167
   macro avg       0.86      0.83      0.85       167
weighted avg       0.87      0.83      0.85       167
 samples avg       0.55      0.53      0.53       167


🔹 Training logistic_regression model...
📊 Classification Report for logistic_regression:
              precision    recall  f1-score   support

        Adhd       0.84      0.83      0.83        70
    Dyslexia       0.92      0.84      0.88        97

   micro avg       0.89      0.83      0.86       167
   macro avg       0.88      0.83      0.86       167
weighted avg       0.89      0.83      0.86       167
 samples avg       0.56      0.53      0.53       167



c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver


🔹 Training knn model...
📊 Classification Report for knn:


...

Epoch 1/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.4616 - loss: 0.6826
Epoch 2/30
48/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5421 - loss: 0.5593

c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\callbacks\early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5425 - loss: 0.5575
Epoch 3/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6028 - loss: 0.4514
Epoch 4/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6668 - loss: 0.3785
Epoch 5/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6456 - loss: 0.3535
Epoch 6/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6576 - loss: 0.3245
Epoch 7/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6676 - loss: 0.3402
Epoch 8/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6231 - loss: 0.3270
Epoch 9/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6504 - loss: 0.3100
Epoch 10/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6714 - loss: 0.3165
Epoch 11/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6560 - loss: 0.2961
Epoch 12/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6524 - loss: 0.3224
Epoch 13/30
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6807 - loss: 0.2945

c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\towqe\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(aver

              precision    recall  f1-score   support

        Adhd       0.85      0.81      0.83        70
    Dyslexia       0.90      0.84      0.87        97

   micro avg       0.88      0.83      0.85       167
   macro avg       0.88      0.82      0.85       167
weighted avg       0.88      0.83      0.85       167
 samples avg       0.55      0.53      0.53       167


✅ All models trained and saved.
